In [ ]:
# paste here any dependices that you need to pip install or that chat/claude tells you to do if you get errors
!pip install torch torchvision torchsummary pandas pillow scikit-learn tqdm
# !pip install torch torchvision torchsummary pandas pillow scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim import Adam
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


In [ ]:
# Only run this block if the master_movie_train/test.csv(s) haven't been created yet
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

BASE_DIR = Path("/content/gdrive/Shareddrives/FML_FINAL/Data")
POSTER_DIR = BASE_DIR / "posters"
RAW_CSV  = BASE_DIR / "master_movie_data.csv"

def prepare_data():
    print("\n--- Running Data Preparation ---")
    df = pd.read_csv(RAW_CSV)
    print(f"Raw rows: {len(df):,}")

    # Fixing Budget typos/Last minute data cleaning, only done once since saving to RAW_CSV
    print("\n Correcting known budget data errors...")

    budget_fixes = [
        (345, 67, 65000000, "Eyes Wide Shut"),
        (48787, 2, 2000000, "Mute Witness"),
        (539, 1999, 806947, "Psycho"),
    ]

    for tmdb_id, old_val, new_val, title in budget_fixes:
        mask = df['tmdb_id'] == tmdb_id
        if mask.any():
            old_budget_raw = df.loc[mask, 'budget_raw'].iloc[0]
            old_budget_2025 = df.loc[mask, 'budget_2025'].iloc[0]

            # Calculate inflation multiplier from existing data
            if old_budget_raw > 0:
                inflation_multiplier = old_budget_2025 / old_budget_raw
            else:
                inflation_multiplier = 1.0

            # Fix budget_raw
            df.loc[mask, 'budget_raw'] = new_val

            # Recalculate budget_2025 with same inflation multiplier
            new_budget_2025 = new_val * inflation_multiplier
            df.loc[mask, 'budget_2025'] = new_budget_2025

            # Recalculate ROI_2025
            gross_2025 = df.loc[mask, 'gross_2025'].iloc[0]
            new_roi = (gross_2025 - new_budget_2025) / new_budget_2025
            df.loc[mask, 'roi_2025'] = new_roi
            df.loc[mask, 'log_roi_2025'] = np.log1p(max(0, new_roi))

            print(f" Fixed {title}")

    # Drop rows with no ROI (needed as target)
    df = df.dropna(subset=["roi_2025"])
    print(f"\nAfter dropping missing roi_2025: {len(df):,}")

    # Vectorized path validation and fixing
    print("Validating poster paths...")

    # Create a mask for valid paths (vectorized check)
    valid_paths = df["poster_file"].notna()

    # For valid entries, check if files exist
    exists_mask = pd.Series([False] * len(df), index=df.index)

    for idx, path in df.loc[valid_paths, "poster_file"].items():
        try:
            exists_mask.loc[idx] = Path(path).exists()
        except (OSError, Exception):
            exists_mask.loc[idx] = False

    # Count how many paths already work
    working_paths = exists_mask.sum()
    print(f" {working_paths:,} paths already valid")

    # For paths that don't exist, try rebuilding with correct base directory
    broken_mask = valid_paths & ~exists_mask
    broken_count = broken_mask.sum()
    # Usually Broken because some may be saved with "drive" rather than "gdrive"
    if broken_count > 0:
        print(f"Attempting to fix {broken_count:,} broken paths...")

        # Vectorized: extract filenames and rebuild paths
        df.loc[broken_mask, "poster_file"] = df.loc[broken_mask, "poster_file"].apply(
            lambda p: str(POSTER_DIR / Path(p).name)
        )

        # Re-check the fixed paths
        for idx, path in df.loc[broken_mask, "poster_file"].items():
            try:
                exists_mask.loc[idx] = Path(path).exists()
            except (OSError, Exception) as e:
                exists_mask.loc[idx] = False

        fixed_count = exists_mask.loc[broken_mask].sum()
        print(f" Fixed {fixed_count:,} paths")

    # Drop rows where posters still don't exist
    df = df[exists_mask]
    dropped = len(valid_paths) - len(df)

    if dropped > 0:
        print(f"Dropped {dropped:,} rows with missing/corrupted posters")

    print(f"After poster validation: {len(df):,} rows")

    # Removing duplicate posters
    before_dedup = len(df)
    df = df.drop_duplicates(subset=['poster_file'], keep='first')
    dedup_removed = before_dedup - len(df)
    if dedup_removed > 0:
        print(f"Removed {dedup_removed:,} duplicate poster entries")
    print(f"After deduplication: {len(df):,} rows with unique posters")

    # Check for any remaining duplicates
    duplicate_check = df[df.duplicated(subset=['poster_file'], keep=False)]
    if len(duplicate_check) > 0:
        print(f"Still found {len(duplicate_check)} duplicates!")
        print(duplicate_check[['tmdb_id', 'title', 'poster_file']].head())
    else:
        print(f"No duplicates remaining")

    # Save cleaned/processed data back to master_movie.csv
    print("\n Saving cleaned data to source file...")
    df.to_csv(RAW_CSV, index=False)
    print(f"master_movie_data.csv updated with all cleaning steps")

    # Split into train and test
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

    # Saved to base directory, so NLP experiment can also use the same train/test split
    train_df.to_csv(BASE_DIR / "master_movie_train.csv", index=False)
    test_df.to_csv(BASE_DIR / "master_movie_test.csv", index=False)

    print(f"\n Saved -> master_movie_train.csv ({len(train_df):,} rows)")
    print(f" Saved -> master_movie_test.csv  ({len(test_df):,} rows)")

    return train_df, test_df

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim import Adam
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

BASE_DIR   = Path("/content/gdrive/Shareddrives/FML_FINAL/Data")
POSTER_DIR = BASE_DIR / "posters"
TRAIN_CSV  = BASE_DIR / "master_movie_train.csv"
TEST_CSV   = BASE_DIR / "master_movie_test.csv"

TARGET_COL = "gross_2025"
USE_LOG_TARGET = True
N_EPOCHS = 50
BATCH_SIZE = 32
LR = 5e-4
NEURONS = 512
LOSS_FN = nn.HuberLoss(delta=1.0)

# Diversity loss to combat regression to mean
USE_DIVERSITY_LOSS = True
DIVERSITY_WEIGHT = 0.05

# Fine-tune ResNet layer4
FINETUNE_RESNET = True

def get_categorical_vocabs(csv_path):
  """Gets unique genres and certifications from a CSV file."""
    df = pd.read_csv(csv_path)
    genres = set()
    if 'genres' in df.columns:
        for g in df['genres'].dropna():
            for x in str(g).split('|'):
                x = x.strip()
                if x:
                    genres.add(x)

    certs = set()
    if 'certification' in df.columns:
        for c in df['certification'].dropna():
            certs.add(str(c).strip())

    return sorted(list(genres)), sorted(list(certs))

class MoviePosterDataset(Dataset):
  """ Custom dataset for movie poster data. """
    def __init__(self, csv_path: Path, genre_vocab: list, cert_vocab: list,
                 budget_stats: dict = None,
                 transform=None, target_col: str = "gross_2025", use_log: bool = True):
        df = pd.read_csv(csv_path)

        def fix_path(p):
            if isinstance(p, str):
                return p.replace("/content/drive/", "/content/gdrive/")
            return p

        df["poster_file"] = df["poster_file"].apply(fix_path)
        df = df.dropna(subset=[target_col])
        df = df[df["poster_file"].apply(lambda p: isinstance(p, str) and Path(p).exists())].reset_index(drop=True)

        self.paths     = df["poster_file"].tolist()
        self.transform = transform

        # # DEBUGGING (how we found some movies said $4 instead of 4 million)
        # print(f"\n Budget data summary:")
        # print(f"  Budget raw min: ${df['budget_2025'].min():,.0f}")
        # print(f"  Budget raw max: ${df['budget_2025'].max():,.0f}")
        # print(f"  Budget raw mean: ${df['budget_2025'].mean():,.0f}")
        # print(f"  Budget raw std: ${df['budget_2025'].std():,.0f}")
        # print(f"  Budget unique values: {df['budget_2025'].nunique()}")


        # Normalizing budget
        df['budget_log'] = np.log1p(df['budget_2025'])

        if budget_stats is None:
            budget_log_mean = df['budget_log'].mean()
            budget_log_std = df['budget_log'].std()
            self.budget_stats = {
                'mean': budget_log_mean,
                'std': budget_log_std
            }
            print(f" Calculated budget stats (TRAINING SET):")
        else:
            budget_log_mean = budget_stats['mean']
            budget_log_std = budget_stats['std']
            self.budget_stats = budget_stats
            print(f" Using provided budget stats (from TRAINING SET):")

        df['budget_normalized'] = (df['budget_log'] - budget_log_mean) / budget_log_std

        # # Budget summary for debugging
        # print(f"Budget log mean: {budget_log_mean:.2f}")
        # print(f"Budget log std: {budget_log_std:.2f}")
        # print(f"Budget normalized min: {df['budget_normalized'].min():.2f}")
        # print(f"Budget normalized max: {df['budget_normalized'].max():.2f}")

        # Defining tabular features
        self.tabular_feats = []
        for _, row in df.iterrows():
            g_feat = [0.0] * len(genre_vocab)
            if pd.notna(row.get('genres')):
                gs = [x.strip() for x in str(row['genres']).split('|')]
                for g in gs:
                    if g and g in genre_vocab:
                        g_feat[genre_vocab.index(g)] = 1.0

            c_feat = [0.0] * len(cert_vocab)
            if pd.notna(row.get('certification')):
                c = str(row['certification']).strip()
                if c in cert_vocab:
                    c_feat[cert_vocab.index(c)] = 1.0

            budget_feat = [row['budget_normalized']]
            self.tabular_feats.append(g_feat + c_feat + budget_feat)

        self.tabular_feats = torch.tensor(self.tabular_feats, dtype=torch.float32)

        raw_labels = df[target_col].astype(float)
        if use_log:
            self.labels = np.log1p(raw_labels.values).astype(np.float32)
        else:
            self.labels = raw_labels.values.astype(np.float32)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
      """ Creates a single sample from the dataset. """
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        tab = self.tabular_feats[idx]
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, tab, label

class AddGaussianNoise:
  """ Custom transform to add Gaussian noise to images. """
    def __init__(self, mean=0.0, std=0.01):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
      """ returns a tensor with Gaussian noise added. """
        return tensor + torch.randn(tensor.size()) * self.std + self.mean

@torch.no_grad()
def extract_features(dl, backbone):
    print("Extracting combined features (Image + Tabular)...")
    backbone.eval()
    t0 = time.time()
    all_features = []
    all_labels   = []
    for x, tab, y in dl:
        x = x.to(device)
        img_feats = torch.flatten(backbone(x), start_dim=1).cpu()
        combined = torch.cat([img_feats, tab], dim=1)
        all_features.append(combined)
        all_labels.append(y)
    print(f"Feature extraction: {time.time() - t0:.1f}s")
    return torch.cat(all_features), torch.cat(all_labels)

def build_regression_mlp(input_dim: int) -> nn.Module:
  """ Builds a regression MLP with 3 hidden layers and dropout. """
    model = nn.Sequential(
        nn.Linear(input_dim, NEURONS), # 512 neurons
        nn.ReLU(),
        nn.Dropout(0.5), # Dropout
        nn.Linear(NEURONS, 256), # 256 neurons
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, 128), # 128 neurons
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 1), # 1 neuron
    ).to(device)
    return model

def train_batch(x, y, model, opt, loss_fn):
  """ Trains single batch of data with optional diversity loss. """
    model.train()
    opt.zero_grad()
    preds = model(x).squeeze(1)

    # Base loss (Huber)
    batch_loss = loss_fn(preds, y)

    # Diversity loss to combat regression to mean
    if USE_DIVERSITY_LOSS:
        pred_std = preds.std()
        target_std = y.std()

        # Penalize if predictions have too low variance
        diversity_loss = -DIVERSITY_WEIGHT * (pred_std / (target_std + 1e-6))
        batch_loss = batch_loss + diversity_loss

    batch_loss.backward()
    opt.step()
    return batch_loss.detach().cpu(), preds.detach().cpu()

@torch.no_grad()
def evaluate(dl, model, loss_fn):
  """ Evaluates the model on a given dataset, using the given loss function. """
    model.eval()
    all_preds, all_labels, losses = [], [], []
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        preds = model(x).squeeze(1)
        losses.append(loss_fn(preds, y).item())
        all_preds.append(preds.cpu())
        all_labels.append(y.cpu())
    preds_t, labels_t = torch.cat(all_preds), torch.cat(all_labels)
    mae = (preds_t - labels_t).abs().mean().item()
    rmse = ((preds_t - labels_t) ** 2).mean().sqrt().item()
    ss_res = ((labels_t - preds_t) ** 2).sum().item()
    ss_tot = ((labels_t - labels_t.mean()) ** 2).sum().item()
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return float(np.mean(losses)), mae, rmse, r2

def train_model(model, train_dl, val_dl=None):
  """ Trains the model using the given training and validation data loaders. """
    opt = Adam(model.parameters(), lr=LR)
    history = {"loss": [], "mae": [], "val_loss": [], "val_mae": [], "val_r2": []}
    t0 = time.time()
    print(f"\n Training ResNet50 + Tabular → Deep MLP")
    print(f"  Model architecture: {NEURONS} → 256 → 128 → 1 (3 hidden layers)")
    print(f"  Dropout: 0.5, 0.4, 0.3")
    print(f"  Learning rate: {LR}")
    print(f"  Diversity loss: {'ENABLED' if USE_DIVERSITY_LOSS else 'DISABLED'} (weight: {DIVERSITY_WEIGHT})")
    print(f"  ResNet fine-tuning: {'ENABLED (layer4)' if FINETUNE_RESNET else 'DISABLED'}")

    best_val_mae = float('inf')
    patience = 10 # Will end training if model doesn't lear after 10 epochs
    patience_counter = 0
    best_model_state = None

    for epoch in tqdm(range(N_EPOCHS), desc="Training Progress"):
        epoch_losses, epoch_preds, epoch_labels = [], [], []
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            loss, preds = train_batch(x, y, model, opt, LOSS_FN)
            epoch_losses.append(loss.item())
            epoch_preds.append(preds)
            epoch_labels.append(y.cpu())
        train_loss = float(np.mean(epoch_losses))
        train_mae = (torch.cat(epoch_preds) - torch.cat(epoch_labels)).abs().mean().item()
        history["loss"].append(train_loss)
        history["mae"].append(train_mae)
        log = f"  Epoch {epoch+1:02d}/{N_EPOCHS} | Train Loss: {train_loss:.4f} | Train MAE: {train_mae:.4f}"

        if val_dl:
            vl, vmae, vrmse, vr2 = evaluate(val_dl, model, LOSS_FN)
            history["val_loss"].append(vl)
            history["val_mae"].append(vmae)
            history["val_r2"].append(vr2)
            log += f" || Val MAE: {vmae:.4f} | Val R²: {vr2:.4f}"

            if vmae < best_val_mae:
                best_val_mae = vmae
                patience_counter = 0
                best_model_state = model.state_dict().copy()
                log += " ★"
            else:
                patience_counter += 1

            if patience_counter >= patience:
                print(f"\n Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
                print(f" Restoring best model (Val MAE: {best_val_mae:.4f})")
                model.load_state_dict(best_model_state)
                break

        print(log)

    print(f"\n Training completed in {time.time() - t0:.1f}s")
    print(f" Best Validation MAE: {best_val_mae:.4f}")
    return history

def visualize_training(history):
    """Plot training curves."""
    epochs = np.arange(1, len(history["loss"]) + 1)
    has_val = len(history["val_loss"]) > 0
    scale_note = "log1p(gross)" if USE_LOG_TARGET else "gross ($)"

    fig, axes = plt.subplots(1, 3 if has_val else 2, figsize=(15, 4))

    axes[0].set_title("Loss (Huber + Diversity)")
    axes[0].plot(epochs, history["loss"], label="train")
    if has_val:
        axes[0].plot(epochs, history["val_loss"], label="val")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].set_title(f"MAE ({scale_note})")
    axes[1].plot(epochs, history["mae"], label="train")
    if has_val:
        axes[1].plot(epochs, history["val_mae"], label="val")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    if has_val:
        axes[2].set_title("R² (val)")
        axes[2].plot(epochs, history["val_r2"], color="green")
        axes[2].axhline(0, color="gray", linestyle="--", linewidth=0.8)
        axes[2].set_xlabel("Epoch")

    plt.tight_layout()
    plt.show()

def run_training_pipeline():
  """ Runs the entire training pipeline. """
    genre_vocab, cert_vocab = get_categorical_vocabs(TRAIN_CSV)
    resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    resnet.fc = nn.Identity()
    resnet = resnet.to(device)

    if FINETUNE_RESNET:
        # Freeze all layers except layer4
        for name, param in resnet.named_parameters():
            if 'layer4' in name:
                param.requires_grad = True
            else:
                param.requires_grad = False
        print("\n ResNet layer4 is UNFROZEN for fine-tuning")
    else:
        # Freeze everything
        for p in resnet.parameters():
            p.requires_grad = False

    base_transform = models.ResNet50_Weights.IMAGENET1K_V1.transforms()

    # Minimal data augmentation: vertical flip (mirror) + small Gaussian noise
    train_transform = transforms.Compose([
        transforms.RandomVerticalFlip(p=0.5), # Vertical flip (mirror effect)
        base_transform, # ImageNet normalization
        AddGaussianNoise(mean=0.0, std=0.01) # Small Gaussian noise
    ])

    test_transform = base_transform

    print("\n MINIMAL data augmentation complete. (vertical flip/mirror + tiny Gaussian noise)")

    full_train_ds = MoviePosterDataset(
        TRAIN_CSV, genre_vocab, cert_vocab,
        budget_stats=None,
        transform=train_transform,
        target_col=TARGET_COL,
        use_log=USE_LOG_TARGET
    )

    budget_stats = full_train_ds.budget_stats

    test_ds = MoviePosterDataset(
        TEST_CSV, genre_vocab, cert_vocab,
        budget_stats=budget_stats,
        transform=test_transform,
        target_col=TARGET_COL,
        use_log=USE_LOG_TARGET
    )

    budget_log_mean = budget_stats['mean']
    budget_log_std = budget_stats['std']

    v_size = int(0.2 * len(full_train_ds))
    t_size = len(full_train_ds) - v_size
    train_ds, val_ds = torch.utils.data.random_split(full_train_ds, [t_size, v_size])

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    train_f, train_l = extract_features(train_dl, resnet)
    val_f, val_l = extract_features(val_dl, resnet)
    test_f, test_l = extract_features(test_dl, resnet)

    train_feat_dl = DataLoader(TensorDataset(train_f, train_l), batch_size=BATCH_SIZE, shuffle=True)
    val_feat_dl = DataLoader(TensorDataset(val_f, val_l), batch_size=BATCH_SIZE, shuffle=False)
    test_feat_dl = DataLoader(TensorDataset(test_f, test_l), batch_size=BATCH_SIZE, shuffle=False)

    mlp = build_regression_mlp(train_f.shape[1])
    history = train_model(mlp, train_feat_dl, val_dl=val_feat_dl)

    test_loss, test_mae, test_rmse, test_r2 = evaluate(test_feat_dl, mlp, LOSS_FN)
    print(f"\nResults:")
    print(f"Test MAE: {test_mae:.4f} (log scale)")
    if USE_LOG_TARGET:
        print(f"expm1({test_mae:.4f}) ≈ ${np.expm1(test_mae):,.0f} off on average")
    print(f"Test R²: {test_r2:.4f}")

    visualize_training(history)

    return mlp, resnet, test_ds, genre_vocab, cert_vocab, budget_log_mean, budget_log_std, history

In [ ]:
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# # Debugging visuazation methods
# def analyze_prediction_errors(y_true, y_pred, budget_values=None, titles=None):
#     """
#     Create comprehensive scatter plots to diagnose prediction errors.
#     Args:
#         y_true: Actual values (log scale) - numpy array
#         y_pred: Predicted values (log scale) - numpy array
#         budget_values: Optional budget values for color coding
#         titles: Optional movie titles for labeling outliers
#     """
#     from scipy import stats
#     from sklearn.metrics import r2_score

#     # Calculate errors
#     errors = y_pred - y_true
#     abs_errors = np.abs(errors)

#     # Create figure with subplots
#     fig, axes = plt.subplots(2, 2, figsize=(16, 12))

#     # Plot 1: Predicted vs Actual (main diagnostic)
#     ax = axes[0, 0]

#     # Perfect prediction line
#     min_val = min(y_true.min(), y_pred.min())
#     max_val = max(y_true.max(), y_pred.max())
#     ax.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.5, linewidth=2, label='Perfect prediction')

#     # Scatter plot
#     if budget_values is not None:
#         scatter = ax.scatter(y_true, y_pred, c=budget_values, cmap='viridis', alpha=0.6, s=50)
#         plt.colorbar(scatter, ax=ax, label='Budget (log scale)')
#     else:
#         ax.scatter(y_true, y_pred, alpha=0.6, s=50)

#     # Calculate R²
#     r2 = r2_score(y_true, y_pred)

#     # Calculate correlation
#     corr, p_value = stats.pearsonr(y_true, y_pred)

#     ax.set_xlabel('Actual log(Gross)', fontsize=12)
#     ax.set_ylabel('Predicted log(Gross)', fontsize=12)
#     ax.set_title(f'Predicted vs Actual\nR² = {r2:.4f}, Corr = {corr:.4f} (p={p_value:.2e})', fontsize=14)
#     ax.legend()
#     ax.grid(True, alpha=0.3)

#     # Add text with summary stats
#     mae = np.mean(abs_errors)
#     rmse = np.sqrt(np.mean(errors**2))
#     ax.text(0.05, 0.95, f'MAE: {mae:.4f}\nRMSE: {rmse:.4f}',
#             transform=ax.transAxes, verticalalignment='top',
#             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

#     # Plot 2: Residual Plot (errors vs predicted)
#     ax = axes[0, 1]
#     ax.scatter(y_pred, errors, alpha=0.6, s=50)
#     ax.axhline(y=0, color='r', linestyle='--', linewidth=2)
#     ax.set_xlabel('Predicted log(Gross)', fontsize=12)
#     ax.set_ylabel('Residual (Predicted - Actual)', fontsize=12)
#     ax.set_title('Residual Plot\n(Check for systematic bias)', fontsize=14)
#     ax.grid(True, alpha=0.3)

#     # Add trend line
#     z = np.polyfit(y_pred, errors, 1)
#     p = np.poly1d(z)
#     ax.plot(sorted(y_pred), p(sorted(y_pred)), "g--", alpha=0.8, linewidth=2,
#             label=f'Trend: y={z[0]:.3f}x+{z[1]:.3f}')
#     ax.legend()

#     # Plot 3: Error Distribution
#     ax = axes[1, 0]
#     ax.hist(errors, bins=50, alpha=0.7, edgecolor='black')
#     ax.axvline(x=0, color='r', linestyle='--', linewidth=2, label='Zero error')
#     ax.axvline(x=np.mean(errors), color='g', linestyle='--', linewidth=2, label=f'Mean: {np.mean(errors):.3f}')
#     ax.axvline(x=np.median(errors), color='orange', linestyle='--', linewidth=2, label=f'Median: {np.median(errors):.3f}')
#     ax.set_xlabel('Error (Predicted - Actual)', fontsize=12)
#     ax.set_ylabel('Frequency', fontsize=12)
#     ax.set_title('Error Distribution\n(Should be centered at 0)', fontsize=14)
#     ax.legend()
#     ax.grid(True, alpha=0.3)

#     # Plot 4: Q-Q Plot (normality check)
#     ax = axes[1, 1]
#     stats.probplot(errors, dist="norm", plot=ax)
#     ax.set_title('Q-Q Plot\n(Check if errors are normally distributed)', fontsize=14)
#     ax.grid(True, alpha=0.3)

#     plt.tight_layout()
#     plt.savefig('prediction_error_analysis.png', dpi=300, bbox_inches='tight')
#     plt.show()

#     # Statistical Summary
#     print("PREDICTION ERROR ANALYSIS")

#     print(f"\nOverall Performance:")
#     print(f" MAE (log scale):     {mae:.4f}")
#     print(f" RMSE (log scale):    {rmse:.4f}")
#     print(f" R² Score:            {r2:.4f}")
#     print(f" Correlation:         {corr:.4f} (p={p_value:.2e})")

#     print(f"\n Error Statistics:")
#     print(f" Mean Error:          {np.mean(errors):.4f}")
#     print(f" Median Error:        {np.median(errors):.4f}")
#     print(f" Std Dev:             {np.std(errors):.4f}")
#     print(f" Min Error:           {np.min(errors):.4f}")
#     print(f" Max Error:           {np.max(errors):.4f}")

#     # Check for bias
#     print(f"\n Bias Detection:")
#     if abs(np.mean(errors)) < 0.1:
#         print(f" Low bias (mean error ≈ 0)")
#     else:
#         bias_direction = "over-predicting" if np.mean(errors) > 0 else "under-predicting"
#         print(f" Model is {bias_direction} on average")

#     # Homoscedasticity check
#     low_pred = y_pred < np.median(y_pred)
#     high_pred = y_pred >= np.median(y_pred)
#     var_low = np.var(errors[low_pred])
#     var_high = np.var(errors[high_pred])
#     var_ratio = var_high / var_low

#     print(f"\n Homoscedasticity (constant variance):")
#     print(f" Variance (low predictions):  {var_low:.4f}")
#     print(f" Variance (high predictions): {var_high:.4f}")
#     print(f" Ratio:                       {var_ratio:.4f}")
#     if 0.8 < var_ratio < 1.2:
#         print(f" Variance is relatively constant")
#     else:
#         print(f" Heteroscedasticity detected (variance changes with prediction level)")

#     # Find worst predictions
#     if titles is not None:
#         print(f"\n Top 10 Worst Predictions:")
#         worst_idx = np.argsort(abs_errors)[-10:][::-1]
#         for i, idx in enumerate(worst_idx, 1):
#             actual_val = y_true[idx]
#             pred_val = y_pred[idx]
#             error = errors[idx]
#             title = titles[idx]
#             print(f"  {i:2d}. {title:40s} | Actual: {actual_val:6.2f} | Pred: {pred_val:6.2f} | Error: {error:+6.2f}")

#         # Best predictions
#         print(f"\n Top 10 Best Predictions:")
#         best_idx = np.argsort(abs_errors)[:10]
#         for i, idx in enumerate(best_idx, 1):
#             actual_val = y_true[idx]
#             pred_val = y_pred[idx]
#             error = errors[idx]
#             title = titles[idx]
#             print(f"  {i:2d}. {title:40s} | Actual: {actual_val:6.2f} | Pred: {pred_val:6.2f} | Error: {error:+6.2f}")
#     return fig

def get_gradcam(mlp, backbone, img_tensor, tab_tensor):
    """Generates a Grad-CAM heatmap showing model attention for gross prediction."""
    target_layer = backbone.layer4
    activations, gradients = [None], [None]

    def fwd_hook(_, __, output): activations[0] = output
    def bwd_hook(_, __, grad_out): gradients[0] = grad_out[0]

    h_fwd = target_layer.register_forward_hook(fwd_hook)
    h_bwd = target_layer.register_full_backward_hook(bwd_hook)

    for p in backbone.parameters(): p.requires_grad_(True)

    img = img_tensor.unsqueeze(0).to(device)
    tab = tab_tensor.unsqueeze(0).to(device)

    with torch.enable_grad():
        feat_flat = torch.flatten(backbone(img), 1)
        combined = torch.cat([feat_flat, tab], dim=1)
        pred = mlp(combined).squeeze()

        backbone.zero_grad()
        mlp.zero_grad()
        pred.backward()

    h_fwd.remove()
    h_bwd.remove()
    for p in backbone.parameters(): p.requires_grad_(False)

    grads, acts = gradients[0], activations[0]
    weights = grads.mean(dim=[2, 3], keepdim=True)
    cam = F.relu((weights * acts).sum(dim=1)).squeeze().detach().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam

def show_gradcam_grid(mlp, backbone, dataset, genre_vocab, cert_vocab,
                      budget_log_mean, budget_log_std,
                      indices=None, n=5, alpha_heatmap=0.45, colormap="jet"):
    """Display GradCAM visualizations with tabular features."""
    if indices is None:
        indices = np.random.choice(len(dataset), size=n, replace=False).tolist()
    else:
        n = len(indices)

    cmap = plt.get_cmap(colormap)
    fig, axes = plt.subplots(n, 3, figsize=(14, 4.5 * n))
    if n == 1:
        axes = [axes]

    mlp.eval()
    backbone.eval()

    for row, idx in enumerate(indices):
        img_tensor, tab_tensor, label = dataset[idx]

        raw_pil = Image.open(dataset.paths[idx]).convert("RGB")
        img_np = np.array(raw_pil) / 255.0

        cam = get_gradcam(mlp, backbone, img_tensor, tab_tensor)
        cam_up = np.array(
            Image.fromarray((cam * 255).astype(np.uint8)).resize(
                raw_pil.size, resample=Image.BILINEAR
            )
        ) / 255.0

        heatmap = cmap(cam_up)[..., :3]
        overlay = np.clip((1 - alpha_heatmap) * img_np + alpha_heatmap * heatmap, 0, 1)

        with torch.no_grad():
            feat_flat = torch.flatten(backbone(img_tensor.unsqueeze(0).to(device)), 1)
            combined = torch.cat([feat_flat, tab_tensor.unsqueeze(0).to(device)], dim=1)
            pred_log = mlp(combined).item()

        to_dollars = np.expm1 if USE_LOG_TARGET else (lambda x: x)
        pred_gross = to_dollars(pred_log)
        true_gross = to_dollars(label.item())

        # Calculate percentage difference
        if true_gross > 0:
            pct_diff = ((pred_gross - true_gross) / true_gross) * 100
            if pct_diff > 0:
                pct_text = f"({pct_diff:.1f}% higher than actual)"
            else:
                pct_text = f"({abs(pct_diff):.1f}% lower than actual)"
        else:
            pct_text = ""

        # Column 1: Original image
        axes[row][0].imshow(img_np)
        axes[row][0].set_title(
            f"Sample #{idx}\n"
            f"True Gross: ${true_gross:>12,.0f}\n"
            f"Pred Gross: ${pred_gross:>12,.0f}\n"
            f"            {pct_text}",
            fontsize=8, family="monospace", loc="left"
        )
        axes[row][0].axis("off")

        # Column 2: GradCAM overlay
        axes[row][1].imshow(overlay)
        sm = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(0, 1))
        plt.colorbar(sm, ax=axes[row][1], fraction=0.03, pad=0.02, label="Attention")
        axes[row][1].set_title("Grad-CAM (Image)", fontsize=9)
        axes[row][1].axis("off")

        # Column 3: Tabular features (Genre + Certification + Budget)
        tab_np = tab_tensor.numpy()
        n_genres = len(genre_vocab)
        n_certs = len(cert_vocab)

        genre_vals = tab_np[:n_genres]
        cert_vals = tab_np[n_genres:n_genres + n_certs]
        budget_normalized = tab_np[n_genres + n_certs]

        # Denormalize budget back to dollars
        budget_log = budget_normalized * budget_log_std + budget_log_mean
        budget_dollars = np.expm1(budget_log)

        active_genres = [genre_vocab[i] for i, v in enumerate(genre_vals) if v > 0]
        active_cert = [cert_vocab[i] for i, v in enumerate(cert_vals) if v > 0]

        tabular_text = "Tabular Features:\n\n"
        tabular_text += f"Genres: {', '.join(active_genres) if active_genres else 'None'}\n\n"
        tabular_text += f"Cert: {active_cert[0] if active_cert else 'None'}\n\n"
        tabular_text += f"Budget: ${budget_dollars:,.0f}"

        axes[row][2].text(0.1, 0.5, tabular_text, fontsize=9,
                         verticalalignment='center', family='monospace',
                         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
        axes[row][2].axis("off")

    plt.suptitle("ResNet50 + Tabular Grad-CAM  ·  Gross Revenue Prediction", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

def analyze_feature_importance(mlp, n_image_feats, genre_vocab, cert_vocab):
    """Analyze which features the MLP considers most important."""
    first_layer = mlp[0]
    weights = first_layer.weight.data.cpu().numpy()
    importance = np.abs(weights).mean(axis=0)

    n_genres = len(genre_vocab)
    n_certs = len(cert_vocab)

    # Split into image vs tabular
    image_importance = importance[:n_image_feats].mean()
    genre_importance = importance[n_image_feats:n_image_feats + n_genres]
    cert_importance = importance[n_image_feats + n_genres:n_image_feats + n_genres + n_certs]
    budget_importance = importance[n_image_feats + n_genres + n_certs]

    print("Feature Importance Analysis")
    print(f"\nImage Features (ResNet50):  {image_importance:.6f} (avg)")
    print(f"\nBudget Feature:             {budget_importance:.6f}")

    print(f"\nTop 10 Genres by Importance:")
    genre_ranks = sorted(zip(genre_vocab, genre_importance), key=lambda x: x[1], reverse=True)
    for i, (genre, imp) in enumerate(genre_ranks[:10], 1):
        print(f"  {i:2d}. {genre:20s}  {imp:.6f}")

    print(f"\nCertifications by Importance:")
    print("-" * 40)
    cert_ranks = sorted(zip(cert_vocab, cert_importance), key=lambda x: x[1], reverse=True)
    for i, (cert, imp) in enumerate(cert_ranks, 1):
        print(f"  {i:2d}. {cert:20s}  {imp:.6f}")

    # Overall comparison
    print(f"\nNet feature Contribution:")
    total_image = image_importance * n_image_feats
    total_genre = genre_importance.sum()
    total_cert = cert_importance.sum()
    total_budget = budget_importance
    total = total_image + total_genre + total_cert + total_budget

    print(f"  Image Features:    {total_image/total*100:5.1f}%")
    print(f"  Budget Feature:    {total_budget/total*100:5.1f}%")
    print(f"  Genre Features:    {total_genre/total*100:5.1f}%")
    print(f"  Cert Features:     {total_cert/total*100:5.1f}%")

def run_visualizations(mlp, resnet, test_ds, test_df, genre_vocab, cert_vocab,
                       budget_log_mean, budget_log_std, n=5):
    """Run GradCAM visualizations and feature importance analysis."""
    print("\n Running Visualizations...")

    # Feature importance analysis
    n_image_feats = 2048  # ResNet50 output
    analyze_feature_importance(mlp, n_image_feats, genre_vocab, cert_vocab)

    # Get predictions AND actual values for all test samples
    with torch.no_grad():
        all_preds = []
        all_actuals = []  # collect actual values
        for img_t, tab_t, label in test_ds:
            feat = torch.flatten(resnet(img_t.unsqueeze(0).to(device)), 1)
            combined = torch.cat([feat, tab_t.unsqueeze(0).to(device)], dim=1)
            p = mlp(combined).item()
            all_preds.append(p)
            all_actuals.append(label.item())

    all_preds = np.array(all_preds)
    all_actuals = np.array(all_actuals)

    # Get budget and titles for better analysis
    budget_log = np.log1p(test_df['budget_2025'].values)
    titles = test_df['title'].values

    # # Debugging: Run error analysis
    # analyze_prediction_errors(
    #     y_true=all_actuals,
    #     y_pred=all_preds,
    #     budget_values=budget_log,
    #     titles=titles
    # )

    # Comment ones you don't wanna see, or do all
    # Get top N predictions
    top_n = np.argsort(all_preds)[-n:][::-1].tolist()
    show_gradcam_grid(mlp, resnet, test_ds, genre_vocab, cert_vocab,
                      budget_log_mean, budget_log_std, indices=top_n)

    # Get lowest N predictions
    lowest_n = np.argsort(all_preds)[:n].tolist()
    show_gradcam_grid(mlp, resnet, test_ds, genre_vocab, cert_vocab,
                      budget_log_mean, budget_log_std, indices=lowest_n)

    # Get random N predictions
    random_n = np.random.choice(len(all_preds), size=n, replace=False).tolist()
    show_gradcam_grid(mlp, resnet, test_ds, genre_vocab, cert_vocab,
                      budget_log_mean, budget_log_std, indices=random_n)

def main():

    train_df, test_df = prepare_data() # Can comment out if master_movies_train/test.csv(s) already created

    # Run training
    mlp, resnet, test_ds, genre_vocab, cert_vocab, budget_log_mean, budget_log_std, history = run_training_pipeline()

    # run Visualizations
    run_visualizations(mlp, resnet, test_ds, test_df, genre_vocab, cert_vocab,
                       budget_log_mean, budget_log_std, n=10)

if __name__ == "__main__":
    main()